# Avaliação de Modelos — CreditGuard AI

Carrega os artefatos gerados por `Model/train.py` e avalia os 5 modelos comparados:
**Dummy · Regressão Logística · Random Forest · XGBoost · LightGBM**

Pré-requisito: executar `python Model/train.py` a partir da raiz do projeto.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

## Carregamento de Artefatos

In [ ]:
with open("Model/config.yaml") as f:
    cfg = yaml.safe_load(f)

arts = cfg["artifacts"]

with open(arts["metadata"], encoding="utf-8") as f:
    metadata = json.load(f)

comparison_df    = pd.read_csv(arts["comparison"])
preprocessor     = joblib.load(arts["preprocessor"])
all_models       = joblib.load(arts["all_models"])
test_preds       = joblib.load(arts["predictions_test"])

y_test           = test_preds["y_test"]
probabilities    = test_preds["proba"]
predictions      = test_preds["pred"]
model_names      = comparison_df["Modelo"].tolist()
feature_names    = preprocessor.get_feature_names_out()

print(f"Melhor modelo : {metadata['melhor_modelo']}")
print(f"ROC-AUC       : {metadata['roc_auc']:.4f}")
print(f"Recall        : {metadata['recall_classe_1']:.4f}")
print(f"Linhas teste  : {len(y_test):,}")
print(f"Features (enc): {len(feature_names):,}")

## Tabela Comparativa — 5 Modelos

In [ ]:
cols_display = [
    "Modelo", "ROC_AUC", "Recall_Classe_1",
    "Precision_Classe_1", "F1_Classe_1",
    "Average_Precision", "Tempo_Treinamento_Segundos",
]
display_df = comparison_df[cols_display].copy()
display_df.index = range(1, len(display_df) + 1)
display_df.columns = [
    "Modelo", "ROC-AUC", "Recall",
    "Precision", "F1",
    "Avg Precision", "Tempo (s)",
]
display_df.style.format({
    "ROC-AUC": "{:.4f}",
    "Recall": "{:.4f}",
    "Precision": "{:.4f}",
    "F1": "{:.4f}",
    "Avg Precision": "{:.4f}",
    "Tempo (s)": "{:.2f}",
}).highlight_max(subset=["ROC-AUC", "Recall", "Precision", "F1"], color="#d4edda")

## Matrizes de Confusão

In [ ]:
n = len(model_names)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))

for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Adimplente", "Inadimplente"],
        yticklabels=["Adimplente", "Inadimplente"],
    )
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{name}\nRecall={tp/(tp+fn):.3f}  Prec={tp/(tp+fp+1e-9):.3f}",
                 fontsize=8)
    ax.set_xlabel("Previsto", fontsize=8)
    ax.set_ylabel("Real", fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle("Matrizes de Confusão — Conjunto de Teste", y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## Curvas ROC — Sobrepostas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name in model_names:
    auc = roc_auc_score(y_test, probabilities[name])
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Aleatório (AUC = 0.5000)")
ax.set_xlabel("Taxa de Falsos Positivos (FPR)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (TPR)")
ax.set_title("Curvas ROC — Comparação de Modelos")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## Curvas Precision-Recall — Sobrepostas

In [ ]:
baseline = float(y_test.mean())

fig, ax = plt.subplots(figsize=(8, 6))

for name in model_names:
    ap = average_precision_score(y_test, probabilities[name])
    prec, rec, _ = precision_recall_curve(y_test, probabilities[name])
    ax.plot(rec, prec, label=f"{name}  (AP = {ap:.4f})")

ax.axhline(baseline, color="k", linestyle="--", linewidth=0.8,
           label=f"Baseline sem skill ({baseline:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curvas Precision-Recall — Comparação de Modelos")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## Interpretabilidade — Feature Importance (Random Forest · XGBoost · LightGBM)

In [ ]:
TREE_MODELS = ["Random Forest", "XGBoost", "LightGBM"]
available = [m for m in TREE_MODELS if m in all_models]

TOP_N = 20
fig, axes = plt.subplots(1, len(available), figsize=(8 * len(available), 9))
if len(available) == 1:
    axes = [axes]

for ax, name in zip(axes, available):
    model = all_models[name]
    imp_df = (
        pd.DataFrame({"Variavel": feature_names, "Importancia": model.feature_importances_})
        .sort_values("Importancia", ascending=False)
        .head(TOP_N)
    )
    imp_df[::-1].set_index("Variavel")["Importancia"].plot(
        kind="barh", ax=ax, color="steelblue", edgecolor="white"
    )
    ax.set_title(f"Top {TOP_N} Features — {name}", fontsize=10)
    ax.set_xlabel("Importância")
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()

## Interpretabilidade — Coeficientes da Regressão Logística

In [ ]:
LR_KEY = "Regressão Logística"
if LR_KEY in all_models:
    lr = all_models[LR_KEY]
    coef = lr.coef_[0]
    coef_df = (
        pd.DataFrame({"Variavel": feature_names, "Coeficiente": coef})
        .assign(Abs=lambda d: d["Coeficiente"].abs())
        .sort_values("Abs", ascending=False)
        .drop(columns="Abs")
        .head(20)
    )

    colors = ["#DC2626" if c > 0 else "#16A34A" for c in coef_df["Coeficiente"]]

    fig, ax = plt.subplots(figsize=(9, 8))
    coef_df[::-1].set_index("Variavel")["Coeficiente"].plot(
        kind="barh", ax=ax, color=colors[::-1], edgecolor="white"
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(
        "Top 20 Coeficientes — Regressão Logística\n"
        "(vermelho = aumenta risco de inadimplência · verde = reduz risco)",
        fontsize=10,
    )
    ax.set_xlabel("Coeficiente")
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    plt.show()
else:
    print(f"Modelo '{LR_KEY}' não encontrado em all_models.joblib")

## Classification Report — Melhor Modelo

In [ ]:
best_name = metadata["melhor_modelo"]
print(f"{'='*60}")
print(f"Melhor modelo: {best_name}")
print(f"{'='*60}")
print(classification_report(
    y_test, predictions[best_name],
    target_names=["Adimplente (0)", "Inadimplente (1)"],
    digits=4, zero_division=0,
))

## Conclusão

In [ ]:
best = metadata["melhor_modelo"]
best_row = comparison_df[comparison_df["Modelo"] == best].iloc[0]

print("Modelo selecionado para produção:")
print(f"  Algoritmo  : {best}")
print(f"  ROC-AUC    : {best_row['ROC_AUC']:.4f}")
print(f"  Recall     : {best_row['Recall_Classe_1']:.4f}  "
      "← métrica prioritária (custo de falso negativo > falso positivo)")
print(f"  Precision  : {best_row['Precision_Classe_1']:.4f}")
print(f"  F1         : {best_row['F1_Classe_1']:.4f}")
print(f"  Features   : {metadata['n_features_encoded']:,} após encoding")
print(f"  Treino     : {metadata['linhas_treino']:,} | Teste: {metadata['linhas_teste']:,}")
print()
print("Ranking final (por ROC-AUC):")
for i, row in comparison_df.iterrows():
    marker = " ← eleito" if row["Modelo"] == best else ""
    print(f"  {i+1}. {row['Modelo']:<25} ROC-AUC={row['ROC_AUC']:.4f}  Recall={row['Recall_Classe_1']:.4f}{marker}")